In [27]:
from langchain.tools import tool

In [28]:
import requests 

In [17]:
from langchain_core.prompts import PromptTemplate

In [8]:
from langchain_tavily import TavilySearch

In [7]:
%pip install --force-reinstall --no-deps python-dotenv

  Using cached python_dotenv-1.2.4-py3-none-any.whl.metadata (29 kB)
Using cached python_dotenv-1.2.4-py3-none-any.whl (23 kB)
  Attempting uninstall: python-dotenv
    Found existing installation: python-dotenv 1.2.4
    Uninstalling python-dotenv-1.2.4:
      Successfully uninstalled python-dotenv-1.2.4
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [1]:
import dotenv
print(dotenv.__file__)


c:\Python\envs\langagent\Lib\site-packages\dotenv\__init__.py


In [2]:
%pip install langchain-community langchain langchain-classic tavily-python


Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install python-dotenv langchain-google-genai certifi

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
print(sys.executable)
%pip show langchain-google-genai

c:\Python\envs\langagent\python.exe
Name: langchain-google-genai
Version: 4.4.2
Summary: An integration package connecting Google's genai package and LangChain
Home-page: https://docs.langchain.com/oss/python/integrations/providers/google
Author: 
Author-email: 
License: MIT
Location: c:\Python\envs\langagent\Lib\site-packages
Requires: filetype, google-genai, langchain-core, pydantic
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import certifi
from dotenv import load_dotenv

#from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_classic import hub

In [4]:
from langchain_classic.agents import create_react_agent, AgentExecutor

In [29]:
#================================
# LOAD ENVIRONMENT VARIABLES
#================================
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

GEMINI_API_KEY=os.getenv("GEMINI_API_KEY")
TAVILY_API_KEY=os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY= os.getenv("WEATHERSTACK_API_KEY")

In [30]:
search_tool=TavilySearch(max_results=2)

In [50]:
@tool
def get_weatheer_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """
    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)
    data = response.json()

    if "current" not in data:
        return f"Weather data not available for {city}."

    # Weatherstack returns a list like ["Patchy rain nearby"]. 
    # Grab the first element safely without character-joining bugs:
    weather_desc = data['current']['weather_descriptions']
    if isinstance(weather_desc, list) and len(weather_desc) > 0:
        desc_str = weather_desc[0]
    else:
        desc_str = str(weather_desc)

    return (
        f"city: {city}\n"
        f"temperature: {data['current']['temperature']}°C\n"
        f"weather_descriptions: {desc_str}\n"
        f"Humidity: {data['current']['humidity']}%"
    )
    

In [32]:
result=search_tool.invoke("Give me the latest news of IND VS WI  17/10/2026")
result

{'query': 'Give me the latest news of IND VS WI  17/10/2026',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://sports.yahoo.com/articles/ind-vs-wi-t20-2026-133912255.html',
   'title': "IND vs WI T20 2026 schedule: Start dates, venues and timings for every T20I match of West Indies' tour of India - Yahoo Sports",
   'content': "### KEY TAKEAWAYS:\n\n Five T20I matches will be played between India and the West Indies from October 6 to October 17, 2026.\n All matches are evening games starting at 7:00 p.m. IST local time and will be played across five different venues.\n Washington Sundar was named in the original squad but has since been withdrawn to prepare for the New Zealand Test series, replaced by Naman Dhir.\n\nAdvertisement\n\nAdvertisement\n\nAdvertisement\n\n## India vs. West Indies T20I 2026 series overview [...] With the ODI series done and dusted, the West Indies now shift focus to a five-match T20I series against India, the reignin

In [51]:
#===============================
# LLM
#===============================
llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",  # Use an OpenRouter model name (or any other free/paid model)
    temperature=0,
    google_api_key=GEMINI_API_KEY   
)

In [52]:
response=llm.invoke("what year is it?")
response

AIMessage(content=[{'type': 'text', 'text': 'It is 2026.', 'extras': {'signature': 'EvoDCvcDAWkUfRNVy6uVv7rL9Wrz5aRXNbeN6Xu6FRlMm/IMPZnAPIbOF+JO8GiEXN+8+N8Rafk81hKKeJBl2QcLzjk6ocWv5ioePEUFzaH56yH3VkMFAeRlNoIXJIrOTKUP3+vj8fKIW2f6Dn/0X93dF5x0Cu/maPJAiTGxqV9N4DJeZFVZD/RQTS8xAoUS/+1qoY82yv4vMItv2Y5HqeSkXo01SzzYmqf8qY2HmJAvwILdKipJgP1c5y0N7lEFJkOZ5SBCu2AOcfZLJ/7loh8gHhz59eTpgclrLLK++Zyxi4D4FUq1KbcduiofcC7wR4wEqABf3MqgfIE4OWersMFVCXmxEE3LKiH0qXxw3OjRnNhNAD+qohpmP3FY8aW3bFJBxfEMgBM7pPUOntQTOkMxZ0bNyM7Dc+5/1VlZ0oaP+f+m16qONibkpbf1dvHPr662Uk7pfCwWGwg1yeBfBXbs0wlhKtDFE85a56NbVh6luMPxc2BtnfII3GS83JMOprRtI2Oy3YLJujlCRmuIFgR6bv79PVLq0CfbZur3r91I3+n37hZrl/Z3IU1fM7dLRhUSpB3pR7UxwV4dxzsJgh5DYrqO19JD0Xj+UyO7nc6CA8n5cJjlJlA8tJ3mCohQiVoB1AZOQE5TIw3IteCxTpfJkKsqSy9UYblfzNk='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.8-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a149eb-ab57-7582-a22e-c5e9bb3c82c3-0', tool_calls=[], inval

In [53]:
# Define the standard ReAct template locally
template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}"""

prompt = PromptTemplate.from_template(template)

In [54]:
#===============================
# CREATE TOOLS
#===============================
tools=[search_tool,get_weatheer_data]

In [55]:
#===============================
# CREATE AGENT
#===============================
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [56]:
#===============================
# EXECUTOR
#===============================
agent_executor = AgentExecutor(
    agent=agent, 
    tools=tools, 
    verbose=True)

In [57]:
#===============================
# RUN
#===============================
response=agent_executor.invoke({
    "input":(
        "The capital of France is Paris. What is the capital of Germany?"
        " and what is the weather in Berlin?" 
    )
})

print(response['output'])



> Entering new AgentExecutor chain...
Action: get_weatheer_data
Action Input: Berlincity: Berlin
temperature: 12°C
weather_descriptions: Patchy rain nearby
Humidity: 88%Thought: I now know the final answer
Final Answer: The capital of Germany is Berlin. The current weather in Berlin is 12°C with patchy rain nearby and a humidity of 88%.

> Finished chain.
The capital of Germany is Berlin. The current weather in Berlin is 12°C with patchy rain nearby and a humidity of 88%.
